In [1]:
notes = [
    {
        "title": "Overfitting",
        "text": "Overfitting happens when a machine learning model memorises the training data instead of learning patterns that generalise to new unseen data."
    },

    {
        "title": "Underfitting",
        "text": "Underfitting happens when a machine learning model is too simple to learn the important patterns in the training data."
    },

    {
        "title": "Cross Validation",
        "text": "Cross validation evaluates a machine learning model by splitting the data into multiple parts and training and testing the model several times."
    },

    {
        "title": "Logistic Regression",
        "text": "Logistic regression is a classification algorithm commonly used to predict categories such as positive or negative."
    },

    {
        "title": "Random Forest",
        "text": "Random forest combines many decision trees and uses their predictions together to produce a more robust prediction."
    },

    {
        "title": "Neural Networks",
        "text": "A neural network contains layers of connected neurons that learn patterns by adjusting weights during training."
    },

    {
        "title": "Embeddings",
        "text": "Embeddings represent words or pieces of text as vectors of numbers so that related meanings can be represented by nearby vectors."
    },

    {
        "title": "TF-IDF",
        "text": "TF-IDF represents text numerically by giving higher importance to words that are important to a document but relatively uncommon across the collection."
    },

    {
        "title": "Bag of Words",
        "text": "Bag of Words represents text using word counts and mostly ignores the order in which words appear."
    },

    {
        "title": "Early Stopping",
        "text": "Early stopping stops neural network training when validation performance stops improving and can help reduce overfitting."
    }
]

print("Knowledge base contains", len(notes), "notes")

Knowledge base contains 10 notes


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

texts = [note["text"] for note in notes]

vectorizer = TfidfVectorizer()

note_vectors = vectorizer.fit_transform(texts)

print("Created vectors:", note_vectors.shape)

Created vectors: (10, 117)


In [5]:
def retrieve(query, k=3):

    query_vector = vectorizer.transform([query])

    scores = cosine_similarity(
        query_vector,
        note_vectors
    )[0]

    best_indices = scores.argsort()[::-1][:k]

    results = []

    for i in best_indices:

        results.append({
            "score": float(scores[i]),
            "title": notes[i]["title"],
            "text": notes[i]["text"]
        })

    return results

In [6]:
question = "Why does a model memorise the training data?"

results = retrieve(question, k=3)

for result in results:

    print(
        round(result["score"], 3),
        "→",
        result["title"]
    )

0.482 → Cross Validation
0.463 → Underfitting
0.453 → Overfitting


In [8]:
def search_notes():

    question = input("\nAsk a question: ")

    results = retrieve(question, k=3)

    print("\n🔎 Relevant notes:\n")

    for result in results:

        print(
            f"[{result['score']:.3f}] "
            f"{result['title']}"
        )

        print(result["text"])
        print()
search_notes()


Ask a question:  what is llm



🔎 Relevant notes:

[0.245] Underfitting
Underfitting happens when a machine learning model is too simple to learn the important patterns in the training data.

[0.231] Logistic Regression
Logistic regression is a classification algorithm commonly used to predict categories such as positive or negative.

[0.000] Bag of Words
Bag of Words represents text using word counts and mostly ignores the order in which words appear.



In [12]:
!uv pip install openai



Using Python 3.14.6 environment at: E:\AI-ML\.venv
Checked 1 package in 27ms


In [14]:
!pip install openai

   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ------------ --------------------------- 0.5/1.7 MB 3.0 MB/s eta 0:00:01
   ------------------------ --------------- 1.0/1.7 MB 2.5 MB/s eta 0:00:01
   ------------------------------------- -- 1.6/1.7 MB 2.6 MB/s eta 0:00:01
   ---------------------------------------- 1.7/1.7 MB 2.4 MB/s  0:00:00
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------- ----------------------------- 0.5/2.1 MB 1.7 MB/s eta 0:00:01
   --------------- ------------------------ 0.8/2.1 MB 1.7 MB/s eta 0:00:01
   -------------------- ------------------- 1.0/2.1 MB 1.5 MB/s eta 0:00:01
   -------------------- ------------------- 1.0/2.1 MB 1.5 MB/s eta 0:00:01
   ------------------------- -------------- 1.3/2.1 MB 1.3 MB/s eta 0:00:01
   ------------------------------ --------- 1.6/2.1 MB 1.3 MB/s eta 0:00:01
   ---------------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\AB\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [34]:
from openai import OpenAI

client = OpenAI(
    api_key="AQ.Ab8RN6IIqscAoEAyytq3pmG_lXvMiBBj_4WfRYvBhnYD1mnqDA"
)

In [35]:
def ask_ai(question):

    # 1. Find relevant notes
    results = retrieve(question, k=3)

    # 2. Build context
    context = "\n\n".join(
        f"{r['title']}: {r['text']}"
        for r in results
    )

    # 3. Give the context to the LLM
    prompt = f"""
You are a helpful AI tutor.

Answer the question using ONLY the course notes below.

If the notes do not contain enough information to answer,
say: "I don't know based on the course notes."

Course notes:
{context}

Question:
{question}
"""

    # 4. Ask the LLM
    response = client.responses.create(
        model="Gemini 3.5 Flash lite",
        input=prompt
    )

    # 5. Return answer
    return response.output_text

In [36]:
answer = ask_ai(
    "What is overfitting and how can we reduce it?"
)

print(answer)

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: AQ.Ab8RN*****************************************nqDA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

In [ ]:
while True:

    question = input("\nYou: ")

    if question.lower() == "exit":
        print("Goodbye!")
        break

    answer = ask_ai(question)

    print("\nAI:", answer)